# Flight Operations Analytics (2018–2022)
CONSTRUCCIÓN DE DATAMART (MODELO ESTRELLA) EN FORMATO PARQUET Y PRUEBA Y COMPARACIÓN CON ORC


## 0) WORKLOAD — Configuración y utilidades

In [6]:
import sys
import os
import subprocess

print("Python:", sys.version)
print("JAVA_HOME:", os.environ.get("JAVA_HOME"))

result = subprocess.run(
    ["java", "-version"],
    capture_output=True,
    text=True
)

print("STDOUT:", result.stdout)
print("STDERR:", result.stderr)

Python: 3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]
JAVA_HOME: C:\Program Files\Eclipse Adoptium\jdk-17.0.11.9-hotspot
STDOUT: 
STDERR: java version "1.8.0_491"
Java(TM) SE Runtime Environment (build 1.8.0_491-b10)
Java HotSpot(TM) Client VM (build 25.491-b10, mixed mode, sharing)



In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import os
import re
import time

# ============================================================
# 0) CREAR SPARK SESSION - WINDOWS LOCAL
# ============================================================

# Ajusta esto si tu Java está en otra ruta
os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-17.0.11.9-hotspot"
os.environ["PATH"] = os.environ["JAVA_HOME"] + r"\bin;" + os.environ["PATH"]

# Opcional, pero recomendado en Windows si tienes winutils.exe
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["PATH"] = os.environ["HADOOP_HOME"] + r"\bin;" + os.environ["PATH"]

spark = (
    SparkSession.builder
    .appName("FlightOperationsAnalytics")
    .master("local[*]")
    .config("spark.driver.host", "localhost")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.ui.enabled", "false")
    .getOrCreate()
)

# ============================================================
# 1) UTILIDADES
# ============================================================

def hr_size(num_bytes: int) -> str:
    size = float(num_bytes)
    for unit in ["B", "KB", "MB", "GB", "TB"]:
        if size < 1024:
            return f"{size:.2f} {unit}"
        size /= 1024
    return f"{size:.2f} PB"


def dir_size(path: str) -> int:
    total = 0

    if not os.path.exists(path):
        return 0

    for root, dirs, files in os.walk(path):
        for file in files:
            file_path = os.path.join(root, file)
            if os.path.exists(file_path):
                total += os.path.getsize(file_path)

    return total


def list_files(path: str):
    if not os.path.exists(path):
        raise FileNotFoundError(f"No existe la ruta: {path}")

    return [
        os.path.join(path, f)
        for f in os.listdir(path)
    ]


def find_col(patterns, columns):
    for p in patterns:
        rx = re.compile(p, re.IGNORECASE)
        for c in columns:
            if rx.fullmatch(c) or rx.search(c):
                return c
    return None


# ============================================================
# 2) DEFINIR RUTAS PARA WINDOWS LOCAL
# ============================================================

SCHEMA = "flights"

# Mejor usar una carpeta explícita, no "."
BASE_VOLUME_PATH = r"C:\data\flights"

KAGGLE_INPUT_PATH = BASE_VOLUME_PATH

PATH_WORKLOAD = os.path.join(BASE_VOLUME_PATH, "workload")
PATH_LANDING = os.path.join(BASE_VOLUME_PATH, "landing", "flights")
PATH_CURATED = os.path.join(BASE_VOLUME_PATH, "curated", "flights")
PATH_FUNCTIONAL = os.path.join(BASE_VOLUME_PATH, "functional", "datamart")

DB_NAME = SCHEMA

# ============================================================
# 3) CREAR CARPETAS SI NO EXISTEN
# ============================================================

os.makedirs(PATH_WORKLOAD, exist_ok=True)
os.makedirs(PATH_LANDING, exist_ok=True)
os.makedirs(PATH_CURATED, exist_ok=True)
os.makedirs(PATH_FUNCTIONAL, exist_ok=True)

# ============================================================
# 4) MOSTRAR CONFIGURACIÓN
# ============================================================

print("Spark version:", spark.version)
print("DB_NAME:", DB_NAME)
print("BASE_VOLUME_PATH:", BASE_VOLUME_PATH)
print("KAGGLE_INPUT_PATH:", KAGGLE_INPUT_PATH)
print("PATH_LANDING:", PATH_LANDING)
print("PATH_CURATED:", PATH_CURATED)
print("PATH_FUNCTIONAL:", PATH_FUNCTIONAL)

### 0.1 Inspección de archivos de entrada (Kaggle)

In [ ]:
display(list_files(KAGGLE_INPUT_PATH))

## 1) LANDING — Ingesta raw

In [ ]:
files = dbutils.fs.ls(KAGGLE_INPUT_PATH)
candidates = [f.path for f in files if f.path.lower().endswith((".csv",".parquet"))]
print("Candidates:")
for c in candidates:
    print("-", c)

main_file = None
if candidates:
    main_file = max([f for f in files if f.path in candidates], key=lambda x: x.size).path

print("\nMain file (heuristic):", main_file)
assert main_file is not None, "No se encontró .csv o .parquet en KAGGLE_INPUT_PATH. Revisa la ruta."

In [ ]:
if main_file.lower().endswith(".csv"):
    df_raw = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(main_file)
    )
else:
    df_raw = spark.read.parquet(main_file)

display(df_raw.limit(20))
print("Raw columns:", len(df_raw.columns))

In [ ]:
landing_path = f"{PATH_LANDING}/flights_raw_parquet"
df_raw.write.mode("overwrite").parquet(landing_path)

print("Landing written:", landing_path)
display(dbutils.fs.ls(landing_path))

## 2) CURATED — Limpieza / estandarización

In [ ]:
df_land = spark.read.parquet(landing_path)
cols = df_land.columns

# Mapeo
mapping = {
    # Core
    "FlightDate": "FlightDate",
    "Airline": "Airline",
    "Operating_Airline": "Operating_Airline",
    "Origin": "Origin",
    "Dest": "Dest",

    # Cancel/Divert
    "Cancelled": "Cancelled",
    "Diverted": "Diverted",
    "DivAirportLandings": "DivAirportLandings",

    # Scheduled / actual times (HHMM)
    "CRSDepTime": "CRSDepTime",
    "DepTime": "DepTime",
    "CRSArrTime": "CRSArrTime",
    "ArrTime": "ArrTime",

    # Delays (prefer minutes columns)
    "DepDelayMinutes": "DepDelayMinutes",
    "ArrDelayMinutes": "ArrDelayMinutes",
    "DepDel15": "DepDel15",
    "ArrDel15": "ArrDel15",
    "DepartureDelayGroups": "DepartureDelayGroups",
    "ArrivalDelayGroups": "ArrivalDelayGroups",

    # Durations / distance
    "AirTime": "AirTime",
    "CRSElapsedTime": "CRSElapsedTime",
    "ActualElapsedTime": "ActualElapsedTime",
    "Distance": "Distance",
    "DistanceGroup": "DistanceGroup",

    # Flight numbers & aircraft
    "Tail_Number": "Tail_Number",
    "Flight_Number_Marketing_Airline": "Flight_Number_Marketing_Airline",
    "Flight_Number_Operating_Airline": "Flight_Number_Operating_Airline",

    # Airport geography (muy útil para BI)
    "OriginCityName": "OriginCityName",
    "OriginState": "OriginState",
    "OriginStateName": "OriginStateName",
    "DestCityName": "DestCityName",
    "DestState": "DestState",
    "DestStateName": "DestStateName",

    # Time blocks (BI)
    "DepTimeBlk": "DepTimeBlk",
    "ArrTimeBlk": "ArrTimeBlk",

    # Calendar columns
    "Year": "Year",
    "Quarter": "Quarter",
    "Month": "Month",
    "DayofMonth": "DayofMonth",
    "DayOfWeek": "DayOfWeek",
}

# Selección
select_expr = []
for canonical, src in mapping.items():
    if src in cols:
        select_expr.append(F.col(src).alias(canonical))
    else:
        select_expr.append(F.lit(None).alias(canonical))

df_cur = df_land.select(*select_expr)

In [ ]:
df_cur = (
    df_cur
    .withColumn("FlightDate", F.to_date("FlightDate"))

    # Normalizar day-of-month
    .withColumn("DayOfMonth", F.coalesce(F.col("DayofMonth").cast("int"), F.dayofmonth("FlightDate")))

    # DayOfWeek ISO: 1=Mon..7=Sun
    .withColumn("DayOfWeekISO", F.dayofweek("FlightDate"))

    #  Year/Quarter/Month
    .withColumn("Year", F.coalesce(F.col("Year").cast("int"), F.year("FlightDate")))
    .withColumn("Quarter", F.coalesce(F.col("Quarter").cast("int"), F.quarter("FlightDate")))
    .withColumn("Month", F.coalesce(F.col("Month").cast("int"), F.month("FlightDate")))

    # Cast numéricos
    .withColumn("Distance", F.col("Distance").cast("double"))
    .withColumn("AirTime", F.col("AirTime").cast("double"))
    .withColumn("CRSElapsedTime", F.col("CRSElapsedTime").cast("double"))
    .withColumn("ActualElapsedTime", F.col("ActualElapsedTime").cast("double"))
    .withColumn("DepDelayMinutes", F.col("DepDelayMinutes").cast("double"))
    .withColumn("ArrDelayMinutes", F.col("ArrDelayMinutes").cast("double"))

    # Flags
    .withColumn("Cancelled", F.col("Cancelled").cast("int"))
    .withColumn("Diverted", F.col("Diverted").cast("int"))
    .withColumn("DepDel15", F.col("DepDel15").cast("int"))
    .withColumn("ArrDel15", F.col("ArrDel15").cast("int"))
)

display(df_cur.limit(20))

In [ ]:
curated_path_parquet = f"{PATH_CURATED}/flights_curated_parquet"
(df_cur.write.mode("overwrite").partitionBy("Year","Month").parquet(curated_path_parquet))

print("Curated written:", curated_path_parquet)
display(dbutils.fs.ls(curated_path_parquet))

## 3) FUNCTIONAL — Datamart dimensional (estrella)

In [ ]:
df_cur_p = spark.read.parquet(curated_path_parquet)
df_cur_p.columns

In [ ]:
df_cur_p = spark.read.parquet(curated_path_parquet)

# HHMM a Hora/minuto (para dim_hora)
def hhmm_to_hour(colname):
    s = F.lpad(F.col(colname).cast("string"), 4, "0")
    return s.substr(1, 2).cast("int")

def hhmm_to_minute(colname):
    s = F.lpad(F.col(colname).cast("string"), 4, "0")
    return s.substr(3, 2).cast("int")

# dim_fecha, normalizar nombres
dim_fecha = (
    df_cur_p
    .select(
        F.col("FlightDate"),
        F.col("Year").cast("int").alias("Year"),
        F.col("Quarter").cast("int").alias("Quarter"),
        F.col("Month").cast("int").alias("Month"),
        F.col("DayofMonth").cast("int").alias("DayOfMonth"),
        F.col("DayOfWeek").cast("int").alias("DayOfWeek"),
    )
    .dropna(subset=["FlightDate"])
    .dropDuplicates(["FlightDate"])
)

dim_fecha = dim_fecha.withColumn(
    "dim_fecha_id",
    F.row_number().over(Window.orderBy("FlightDate"))
)

# dim_hora: HHMM + Hora/minuto + bloque opcioal
dim_hora = (
    df_cur_p
    .select(
        F.col("CRSDepTime").alias("hhmm"),
        hhmm_to_hour("CRSDepTime").alias("Hora"),
        hhmm_to_minute("CRSDepTime").alias("Minuto"),
        F.col("DepTimeBlk").cast("string").alias("TimeBlock")
    )
    .unionByName(
        df_cur_p.select(
            F.col("CRSArrTime").alias("hhmm"),
            hhmm_to_hour("CRSArrTime").alias("Hora"),
            hhmm_to_minute("CRSArrTime").alias("Minuto"),
            F.col("ArrTimeBlk").cast("string").alias("TimeBlock")
        ),
        allowMissingColumns=True
    )
    .dropna(subset=["hhmm"])
    .dropDuplicates(["hhmm"])
)

dim_hora = dim_hora.withColumn(
    "dim_hora_id",
    F.row_number().over(Window.orderBy("hhmm"))
)

# dim_aerolinea
dim_aerolinea = (
    df_cur_p
    .select(
        F.col("Airline").alias("Airline")
    )
    .dropna(subset=["Airline"])
    .dropDuplicates(["Airline"])
)

dim_aerolinea = dim_aerolinea.withColumn(
    "dim_aerolinea_id",
    F.row_number().over(Window.orderBy("Airline"))
)

# dim_origen / dim_destino (con city/state
dim_origen = (
    df_cur_p
    .select(
        F.col("Origin").alias("Origin"),
        F.col("OriginCityName").alias("OriginCityName"),
        F.col("OriginState").alias("OriginState"),
        F.col("OriginStateName").alias("OriginStateName"),
    )
    .dropna(subset=["Origin"])
    .dropDuplicates(["Origin"])
)
dim_origen = dim_origen.withColumn(
    "dim_origen_id",
    F.row_number().over(Window.orderBy("Origin"))
)

dim_destino = (
    df_cur_p
    .select(
        F.col("Dest").alias("Dest"),
        F.col("DestCityName").alias("DestCityName"),
        F.col("DestState").alias("DestState"),
        F.col("DestStateName").alias("DestStateName"),
    )
    .dropna(subset=["Dest"])
    .dropDuplicates(["Dest"])
)
dim_destino = dim_destino.withColumn(
    "dim_destino_id",
    F.row_number().over(Window.orderBy("Dest"))
)

# dim_ruta
dim_ruta = (
    df_cur_p
    .select(
        F.col("Origin").alias("Origin"),
        F.col("Dest").alias("Dest"),
        F.col("DistanceGroup").cast("int").alias("DistanceGroup"),
    )
    .dropna(subset=["Origin", "Dest"])
    .dropDuplicates(["Origin", "Dest"])
)

dim_ruta = dim_ruta.withColumn(
    "dim_ruta_id",
    F.row_number().over(Window.orderBy("Origin", "Dest"))
)

# dim_avion
dim_avion = (
    df_cur_p
    .select(F.col("Tail_Number").alias("Tail_Number"))
    .dropna(subset=["Tail_Number"])
    .dropDuplicates(["Tail_Number"])
)
dim_avion = dim_avion.withColumn(
    "dim_avion_id",
    F.row_number().over(Window.orderBy("Tail_Number"))
)

# dim_vuelo
dim_vuelo = (
    df_cur_p
    .select(
        F.col("Flight_Number_Marketing_Airline").alias("Flight_Number_Marketing_Airline"),
        F.col("Flight_Number_Operating_Airline").alias("Flight_Number_Operating_Airline"),
    )
    .dropDuplicates(["Flight_Number_Marketing_Airline", "Flight_Number_Operating_Airline"])
)

dim_vuelo = dim_vuelo.withColumn(
    "dim_vuelo_id",
    F.row_number().over(
        Window.orderBy(
            F.coalesce(F.col("Flight_Number_Marketing_Airline"), F.lit("")),
            F.coalesce(F.col("Flight_Number_Operating_Airline"), F.lit("")),
        )
    )
)


print("dim_fecha:", dim_fecha.count())
print("dim_hora:", dim_hora.count())
print("dim_aerolinea:", dim_aerolinea.count())
print("dim_origen:", dim_origen.count())
print("dim_destino:", dim_destino.count())
print("dim_ruta:", dim_ruta.count())
print("dim_avion:", dim_avion.count())
print("dim_vuelo:", dim_vuelo.count())

# FACT
base = df_cur_p

fact = (
    base
    .join(dim_fecha.select("FlightDate", "dim_fecha_id"), on="FlightDate", how="left")
    .join(dim_aerolinea.select("Airline", "dim_aerolinea_id"), on="Airline", how="left")
    .join(dim_origen.select("Origin", "dim_origen_id"), on="Origin", how="left")
    .join(dim_destino.select("Dest", "dim_destino_id"), on="Dest", how="left")
    .join(dim_ruta.select("Origin", "Dest", "dim_ruta_id"), on=["Origin", "Dest"], how="left")
    .join(dim_avion.select("Tail_Number", "dim_avion_id"), on="Tail_Number", how="left")
    .join(
        dim_vuelo.select(
            "Flight_Number_Marketing_Airline",
            "Flight_Number_Operating_Airline",
            "dim_vuelo_id"
        ),
        on=["Flight_Number_Marketing_Airline", "Flight_Number_Operating_Airline"],
        how="left",
    )
)

dh_dep = dim_hora.select(
    F.col("hhmm").alias("CRSDepTime"),
    F.col("dim_hora_id").alias("dim_hora_dep_id")
)
dh_arr = dim_hora.select(
    F.col("hhmm").alias("CRSArrTime"),
    F.col("dim_hora_id").alias("dim_hora_arr_id")
)

fact = (
    fact
    .join(dh_dep, on="CRSDepTime", how="left")
    .join(dh_arr, on="CRSArrTime", how="left")
)

fact = fact.withColumn(
    "fact_id",
    F.row_number().over(Window.orderBy(F.monotonically_increasing_id()))
)


fact_vuelos = fact.select(
    "fact_id",
    "dim_fecha_id",
    F.col("dim_hora_dep_id").alias("dim_hora_id"),       # salida programada (compat README)
    F.col("dim_hora_arr_id").alias("dim_hora_arr_id"),   # llegada programada (extra BI)

    "dim_avion_id",
    "dim_vuelo_id",
    "dim_origen_id",
    "dim_destino_id",
    "dim_ruta_id",
    "dim_aerolinea_id",

    # Medidas
    F.col("Distance").cast("double").alias("Distance"),
    F.col("AirTime").cast("double").alias("AirTime"),
    F.col("CRSElapsedTime").cast("double").alias("CRSElapsedTime"),
    F.col("ActualElapsedTime").cast("double").alias("ActualElapsedTime"),

    # Retrasos
    F.col("DepDelayMinutes").cast("double").alias("DepDelayMinutes"),
    F.col("ArrDelayMinutes").cast("double").alias("ArrDelayMinutes"),

    # REVISAR
    F.col("DepDel15").cast("int").alias("DepDel15"),
    F.col("ArrDel15").cast("int").alias("ArrDel15"),
    F.col("Cancelled").cast("int").alias("Cancelled"),
    F.col("Diverted").cast("int").alias("Diverted"),
    F.col("DepartureDelayGroups").cast("int").alias("DepartureDelayGroups"),
    F.col("ArrivalDelayGroups").cast("int").alias("ArrivalDelayGroups"),

    #
    F.col("Operating_Airline").alias("Operating_Airline"),
    F.col("DepTimeBlk").cast("string").alias("DepTimeBlk"),
    F.col("ArrTimeBlk").cast("string").alias("ArrTimeBlk"),

)

print("fact_vuelos:", fact_vuelos.count())
display(fact_vuelos.limit(20))


In [ ]:
# Escribir functional Parquet (sin dim_cancelacion)
tables = {
    "dim_fecha": dim_fecha,
    "dim_hora": dim_hora,
    "dim_aerolinea": dim_aerolinea,
    "dim_origen": dim_origen,
    "dim_destino": dim_destino,
    "dim_ruta": dim_ruta,
    "dim_avion": dim_avion,
    "dim_vuelo": dim_vuelo,
    "fact_vuelos": fact_vuelos,
}

for name, df_t in tables.items():
    p = f"{PATH_FUNCTIONAL}/parquet/{name}"
    (
        df_t.write
        .mode("overwrite")
        .parquet(p)
    )
    print("Wrote:", p)

display(dbutils.fs.ls(f"{PATH_FUNCTIONAL}/parquet"))

## 4) ORC — Conversión + comparación (tamaño y benchmark)

In [ ]:
# ORC para curated y fact
curated_path_orc = f"{PATH_CURATED}/flights_curated_orc"
df_cur_p.write.mode("overwrite").format("orc").save(curated_path_orc)

fact_orc_path = f"{PATH_FUNCTIONAL}/orc/fact_vuelos"
fact_vuelos.write.mode("overwrite").format("orc").save(fact_orc_path)

# Tamaños
sz_cur_parq  = dir_size(curated_path_parquet)
sz_cur_orc   = dir_size(curated_path_orc)
sz_fact_parq = dir_size(f"{PATH_FUNCTIONAL}/parquet/fact_vuelos")
sz_fact_orc  = dir_size(fact_orc_path)

print("=== SIZE COMPARISON ===")
print("Curated Parquet:", hr_size(sz_cur_parq))
print("Curated ORC   :", hr_size(sz_cur_orc))
print("Fact Parquet  :", hr_size(sz_fact_parq))
print("Fact ORC      :", hr_size(sz_fact_orc))

In [ ]:
def benchmark_read_agg(read_fn, label: str, runs: int = 3):
    times = []
    for _ in range(runs):
        t0 = time.time()
        df_ = read_fn()
        _ = (
            df_
            .groupBy("dim_aerolinea_id")
            .agg(F.avg("DepDelayMinutes").alias("avg_depdelay_min"))
            .orderBy("dim_aerolinea_id")
            .limit(50)
            .collect()
        )

        times.append(time.time() - t0)

    print(label, "times:", [round(x, 3) for x in times], "avg:", round(sum(times) / len(times), 3))


parq_path = f"{PATH_FUNCTIONAL}/parquet/fact_vuelos"
orc_path  = f"{PATH_FUNCTIONAL}/orc/fact_vuelos"

benchmark_read_agg(lambda: spark.read.parquet(parq_path), "FACT Parquet")
benchmark_read_agg(lambda: spark.read.orc(orc_path), "FACT ORC")

## 5) KPI de validación: OTP (DepDelay <= 15)

In [ ]:
fact_df = spark.read.parquet(parq_path)

otp = (
    fact_df
    # Si DepDel15 viene con nulls, los tratamos como 0 (no delayed) para OTP general.
    .withColumn("DepDel15", F.coalesce(F.col("DepDel15").cast("int"), F.lit(0)))
    .agg(
        F.count("fact_id").alias("n_flights"),
        # OTP = porcentaje NO retrasado >15
        F.avg((1 - F.col("DepDel15")).cast("double")).alias("otp_rate_dep"),
        # Retraso promedio de salida
        F.avg(F.col("DepDelayMinutes").cast("double")).alias("avg_depdelay_min"),
        # porcentaj de vuelos retrasados >15
        F.avg(F.col("DepDel15").cast("double")).alias("pct_delayed15_dep"),
    )
)

display(otp)